# DNAffinity: exploratory analysis of sequence descriptors and SELEX affinity

**Final EDA notebook · Stage 1 validated results**

This notebook consolidates the essential descriptor, correlation, PCA and Shape–Affinity results for 11 transcription factors (TFs). It uses the canonical clean feature definitions in `src/feature_config.py` and the validated outputs under `outputs/eda/`.

**Research question.** Do DNA structural and physicochemical descriptors retain associations with SELEX-derived Affinity after accounting for sequence composition?

The analyses are exploratory: correlations are not causal effects, PCA is not a predictive test, and no predictive models or significance claims are made.

## 1. Scope and scientific interpretation

HT-SELEX Affinity is an enrichment-derived estimate, not an independently measured dissociation constant (`Kd`). `ObservedCount` is related to the Affinity calculation, while `SE` describes its uncertainty; neither is treated as a second independent affinity outcome. We examine composition-adjusted descriptor associations, not predictive gain. The complete analysis scripts and output tables remain available for reproducibility. If `outputs/eda/` is absent in a fresh checkout, run `python scripts/eda_shape_affinity_adjusted.py`, `python scripts/eda_cross_feature_correlations.py` and `python scripts/eda_pca_clean_features.py` from the repository root before running this notebook.

In [3]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "src" / "feature_config.py").exists())
sys.path.insert(0, str(ROOT))
from src.feature_config import (
    FEATURE_GROUPS_CLEAN_TETRAMER, FEATURE_GROUPS_CLEAN_HEXAMER,
    FEATURES_CLEAN_TETRAMER, FEATURES_CLEAN_HEXAMER,
)
EDA = ROOT / "outputs" / "eda"
SHAPE = EDA / "shape_affinity"
CROSS = EDA / "cross_feature"
PCA = EDA / "pca"
FIGURES = ROOT / "notebooks" / "plots_EDA"


## 2. Data and preprocessing

The canonical tables contain all possible 4-mers (256) and 6-mers (4,096). A 10-mer contributes seven overlapping tetramer windows and five hexamer windows. In the Shape–Affinity analysis, descriptor values were joined by sequence identifier, averaged across the overlapping windows, and then associated with each TF's SELEX labels. GC content is C + G and was used for description, not as an additional independent adjustment covariate.

In [5]:
validation = pd.read_csv(SHAPE / "sample_sizes_and_validation.csv")
pca_validation = pd.read_csv(PCA / "canonical_input_validation.csv")
assert len(validation) == 11
assert (validation[["Duplicate_Kmers", "Invalid_or_wrong_length_Kmers", "Missing_target_values"]] == 0).all().all()
assert validation["Canonical_predictor_count"].eq(76).all()
assert validation["Tetramer_windows_per_sequence"].eq(7).all()
assert validation["Hexamer_windows_per_sequence"].eq(5).all()
display(pca_validation)
display(validation[["TranscriptionFactor", "Cycle", "N_sequences", "Count_ge_2_retained", "Count_ge_2_removed"]])


,Sequence_length,Original_rows,Clean_rows,Canonical_feature_count,Sequence_ID_sets_match,Values_match_exactly_by_sequence,Original_missing_values,Clean_missing_values
0,tetramer,256,256,36,True,True,0,0
1,hexamer,4096,4096,40,True,True,0,0


,TranscriptionFactor,Cycle,N_sequences,Count_ge_2_retained,Count_ge_2_removed
0,Gata4,4,126624,60155,66469
1,Zfp24,5,406165,226570,179595
2,arid3,6,273249,144479,128770
3,hoxd1,3,887634,662991,224643
4,nkx3,6,963575,848128,115447
5,pax3-like,5,427916,226515,201401
6,six1,5,944350,792523,151827
7,tbx15,5,25991,13599,12392
8,telf,5,691492,404159,287333
9,uncxb,5,234129,128267,105862


The clean tables preserve the original values and sequence identifiers. Stage 1 validation found no missing or duplicated 10-mers, invalid sequence labels, or missing target values among the selected TF-cycle inputs. Exact feature-level comparison against the original tables was performed by sequence ID. The two clean descriptor tables contain 36 tetramer and 40 hexamer features.

In [7]:
tetra = pd.read_csv(ROOT / "data/selex/tetramer/input_tetramer_clean.csv")
hexa = pd.read_csv(ROOT / "data/selex/hexamer/input_hexamer_clean.csv")
assert tetra.columns.tolist() == ["SEQUENCE", *FEATURES_CLEAN_TETRAMER]
assert hexa.columns.tolist() == ["SEQUENCE", *FEATURES_CLEAN_HEXAMER]
assert len(tetra) == tetra.SEQUENCE.nunique() == 256
assert len(hexa) == hexa.SEQUENCE.nunique() == 4096
assert not tetra.isna().any().any() and not hexa.isna().any().any()
display(pd.DataFrame({"Table": ["Tetramer", "Hexamer"], "Sequences": [len(tetra), len(hexa)], "Canonical features": [len(FEATURES_CLEAN_TETRAMER), len(FEATURES_CLEAN_HEXAMER)]}))


,Table,Sequences,Canonical features
0,Tetramer,256,36
1,Hexamer,4096,40


## 3. Descriptor distributions

The next figures show representative distributions from the original descriptor exploration. FCE diagonal and off-diagonal elements are separated because they represent distinct positions in the stiffness matrix. Ion-related variables are retained under their documented feature names; no additional physical interpretation or units are assumed.

In [9]:
display(Image(filename=str(FIGURES / "avg_distributions.png")))

In [10]:
display(Image(filename=str(FIGURES / "fce_diagonal_distributions.png")))
display(Image(filename=str(FIGURES / "fce_off_diagonal_distributions.png")))

In [11]:
display(Image(filename=str(FIGURES / "electrostatic_distributions.png")))
display(Image(filename=str(FIGURES / "hexamer_ion_distributions.png")))

These plots describe the descriptor value ranges across all possible short DNA words. They do not show TF binding outcomes. Distribution shapes can motivate later checks for scaling and redundancy, but do not by themselves establish biological function.

## 4. Exact redundancy and cross-feature relationships

The canonical feature configuration removes mirrored lower-triangular FCE entries, retaining the 21 independent upper-triangular elements. The exact electrostatic identities are `e3 = -e2` and `e5 = -e1 - e4`; therefore the canonical electrostatic set is `e1`, `e2` and `e4`. Strong but imperfect correlations are described separately and are not used to remove features.

In [14]:
group_counts = pd.DataFrame([
    {"Sequence length": k, "Descriptor group": group, "Features": len(names)}
    for k, groups in [("Tetramer", FEATURE_GROUPS_CLEAN_TETRAMER), ("Hexamer", FEATURE_GROUPS_CLEAN_HEXAMER)]
    for group, names in groups.items()
])
display(group_counts.pivot(index="Descriptor group", columns="Sequence length", values="Features"))


Sequence length,Hexamer,Tetramer
Descriptor group,,
AVG,6,6
Electrostatic,3,3
FCE,21,21
Ion-related,10,6


Pearson and Spearman correlations were computed within each clean descriptor table. The following summary counts cross-group pairs with `|Pearson r| ≥ 0.7`. This threshold identifies strong linear associations for description; it is not a test of exact dependence.

In [16]:
pairs = pd.read_csv(CROSS / "group_pair_summary.csv")
cross_pairs = pairs[(pairs.Method == "Pearson") & (pairs.Group_1 != pairs.Group_2)].copy()
display(cross_pairs[["Sequence_length", "Group_1", "Group_2", "Strong_pairs", "Possible_pairs", "Strong_pair_fraction", "Maximum_absolute_correlation"]].sort_values(["Sequence_length", "Strong_pairs"], ascending=[True, False]))
strong = pd.read_csv(CROSS / "strong_feature_pairs.csv")
top_cross = strong[(strong.Method == "Pearson") & (strong.Relationship == "Cross-group")].sort_values("Absolute_correlation", ascending=False).groupby("Sequence_length").head(5)
display(top_cross[["Sequence_length", "Feature_1", "Feature_2", "Group_1", "Group_2", "Correlation"]])


,Sequence_length,Group_1,Group_2,Strong_pairs,Possible_pairs,Strong_pair_fraction,Maximum_absolute_correlation
22,hexamer,AVG,FCE,6,126,0.047619,0.836799
23,hexamer,AVG,Ion-related,1,60,0.016667,0.847766
21,hexamer,AVG,Electrostatic,0,18,0.000000,0.227911
25,hexamer,Electrostatic,Ion-related,0,30,0.000000,0.189218
26,hexamer,FCE,Electrostatic,0,63,0.000000,0.590254
28,hexamer,FCE,Ion-related,0,210,0.000000,0.624847
2,tetramer,AVG,FCE,9,126,0.071429,0.857037
3,tetramer,AVG,Ion-related,3,36,0.083333,0.874714
1,tetramer,AVG,Electrostatic,0,18,0.000000,0.191739
5,tetramer,Electrostatic,Ion-related,0,18,0.000000,0.252187


,Sequence_length,Feature_1,Feature_2,Group_1,Group_2,Correlation
54,tetramer,RISE,m2,AVG,Ion-related,-0.874714
56,tetramer,TILT,SHIFT.SLIDE,AVG,FCE,-0.857037
2,hexamer,RISE,m2,AVG,Ion-related,-0.847766
57,tetramer,SHIFT,SHIFT.SLIDE,AVG,FCE,-0.847415
58,tetramer,ROLL,ROLL.ROLL,AVG,FCE,-0.846811
4,hexamer,TILT,SHIFT.SLIDE,AVG,FCE,-0.836799
5,hexamer,SHIFT,SHIFT.SLIDE,AVG,FCE,-0.830075
60,tetramer,ROLL,RISE.RISE,AVG,FCE,-0.826816
7,hexamer,ROLL,RISE.ROLL,AVG,FCE,-0.810544
8,hexamer,ROLL,ROLL.ROLL,AVG,FCE,-0.809127


In [17]:
display(Image(filename=str(CROSS / "heatmaps" / "tetramer_pearson_all_features.png")))
display(Image(filename=str(CROSS / "heatmaps" / "hexamer_pearson_all_features.png")))

The most frequent strong cross-group relationships involve AVG–FCE and AVG–ion-related pairs. The largest cross-group Pearson pair is RISE–m2 (r = −0.875 for tetramers; −0.848 for hexamers). Such correlations indicate shared variation for those feature pairs, but do not imply complete redundancy. The small electrostatic within-group denominator (three possible pairs) should be kept in view.

## 5. Principal component analysis

Individual descriptor-group and combined PCA were applied separately to tetramer and hexamer tables after standardising each feature with `StandardScaler`. The corrected canonical analyses use the feature lists from `src/feature_config.py`. Reported component counts are the minimum numbers needed to reach 80%, 90% and 95% cumulative variance. PCA summarises variation in descriptor space and does not test affinity association.

In [20]:
from sklearn.decomposition import PCA as SKPCA
from sklearn.preprocessing import StandardScaler

pca_summary = pd.read_csv(PCA / "pca_summary.csv")
individual_rows = []
for length, data, groups in [
    ("tetramer", tetra, FEATURE_GROUPS_CLEAN_TETRAMER),
    ("hexamer", hexa, FEATURE_GROUPS_CLEAN_HEXAMER),
]:
    for group, features in groups.items():
        model = SKPCA().fit(StandardScaler().fit_transform(data[features]))
        cumulative = np.cumsum(model.explained_variance_ratio_)
        individual_rows.append({
            "Sequence_length": length, "Configuration": group, "N_features": len(features),
            "PC1_variance": model.explained_variance_ratio_[0],
            "PC2_variance": model.explained_variance_ratio_[1],
            "PCs_for_80_percent": int(np.searchsorted(cumulative, 0.80) + 1),
            "PCs_for_90_percent": int(np.searchsorted(cumulative, 0.90) + 1),
            "PCs_for_95_percent": int(np.searchsorted(cumulative, 0.95) + 1),
        })
individual_pca_summary = pd.DataFrame(individual_rows)
combined_pca_summary = pca_summary.assign(Analysis="Combined groups")
individual_pca_summary["Analysis"] = "Individual group"
all_pca_summary = pd.concat([individual_pca_summary, combined_pca_summary], ignore_index=True)
display(all_pca_summary[["Sequence_length", "Analysis", "Configuration", "N_features", "PC1_variance", "PCs_for_80_percent", "PCs_for_90_percent", "PCs_for_95_percent"]])
assert pca_summary.Feature_names_match_canonical.all()


,Sequence_length,Analysis,Configuration,N_features,PC1_variance,PCs_for_80_percent,PCs_for_90_percent,PCs_for_95_percent
0,tetramer,Individual group,AVG,6,0.355044,3,4,4
1,tetramer,Individual group,FCE,21,0.286968,6,8,10
2,tetramer,Individual group,Electrostatic,3,0.666667,2,2,3
3,tetramer,Individual group,Ion-related,6,0.363556,3,4,5
4,hexamer,Individual group,AVG,6,0.350198,3,4,4
5,hexamer,Individual group,FCE,21,0.283818,6,9,11
6,hexamer,Individual group,Electrostatic,3,0.666667,2,2,3
7,hexamer,Individual group,Ion-related,10,0.222027,5,7,9
8,tetramer,Combined groups,Electrostatic,3,0.666667,2,2,3
9,tetramer,Combined groups,AVG + FCE,27,0.283701,6,9,12


In [21]:
display(Image(filename=str(PCA / "figures" / "pca_tetramer_avg_fce_electrostatic_ionrelated_cumulative_variance.png")))
display(Image(filename=str(PCA / "figures" / "pca_hexamer_avg_fce_electrostatic_ionrelated_cumulative_variance.png")))

In [22]:
display(Image(filename=str(PCA / "figures" / "pca_tetramer_avg_fce_electrostatic_ionrelated_scores.png")))
display(Image(filename=str(PCA / "figures" / "pca_hexamer_avg_fce_electrostatic_ionrelated_scores.png")))

In [23]:
display(Image(filename=str(PCA / "figures" / "pca_tetramer_avg_fce_electrostatic_ionrelated_loadings.png")))
display(Image(filename=str(PCA / "figures" / "pca_hexamer_avg_fce_electrostatic_ionrelated_loadings.png")))


For the all-group configuration, 8 tetramer PCs and 10 hexamer PCs account for 80% of variance; 11 and 15 PCs reach 90%, while 15 and 20 reach 95%. The loading heatmaps show how the standardized descriptors contribute to each component; component signs are arbitrary. The score plots show projections of the descriptor tables, not clusters of binding affinity. Earlier notebook electrostatic and combined PCA runs included dependent electrostatic columns; the corrected results shown here use the canonical three-column electrostatic set.

## 6. HT-SELEX outcome coverage

The 11 TF datasets use the selected cycles recorded in the validation table. `ObservedCount` is strongly related to the derivation of Affinity, so it is used only in a sensitivity filter. `SE` is preserved and validated but is not analysed as an independent outcome. The existing plot gives a compact view of the observed count and Affinity distributions by TF.

In [26]:
display(validation[["TranscriptionFactor", "Cycle", "N_sequences", "Count_ge_2_retained", "Count_ge_2_removed", "Count_ge_2_removed_fraction"]].style.format({"Count_ge_2_removed_fraction": "{:.1%}"}))
display(Image(filename=str(FIGURES / "selex_count_affinity_boxplots.png")))

,TranscriptionFactor,Cycle,N_sequences,Count_ge_2_retained,Count_ge_2_removed,Count_ge_2_removed_fraction
0,Gata4,4,126624,60155,66469,52.5%
1,Zfp24,5,406165,226570,179595,44.2%
2,arid3,6,273249,144479,128770,47.1%
3,hoxd1,3,887634,662991,224643,25.3%
4,nkx3,6,963575,848128,115447,12.0%
5,pax3-like,5,427916,226515,201401,47.1%
6,six1,5,944350,792523,151827,16.1%
7,tbx15,5,25991,13599,12392,47.7%
8,telf,5,691492,404159,287333,41.6%
9,uncxb,5,234129,128267,105862,45.2%


In [27]:
# Stream through each TF file and sample SE values for a compact distribution comparison.
TF_CYCLES = {
    "Gata4": 4, "Zfp24": 5, "arid3": 6, "hoxd1": 3, "nkx3": 6,
    "pax3-like": 5, "six1": 5, "tbx15": 5, "telf": 5, "uncxb": 5, "xbpc": 5,
}
se_samples = {}
for tf, cycle in TF_CYCLES.items():
    path = ROOT / "data" / "selex" / "transcription_factors" / tf / f"{cycle}.txt"
    sampled_chunks = []
    for chunk_number, chunk in enumerate(pd.read_csv(path, sep="\t", usecols=["SE"], chunksize=100_000)):
        values = chunk["SE"].dropna()
        if not values.empty:
            sampled_chunks.append(values.sample(n=min(250, len(values)), random_state=cycle + chunk_number))
    se_samples[tf] = pd.concat(sampled_chunks).sample(n=min(4000, len(pd.concat(sampled_chunks))), random_state=cycle)

fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot([se_samples[tf] for tf in TF_CYCLES], labels=list(TF_CYCLES), showfliers=False, patch_artist=True,
           boxprops={"facecolor": "#9ecae1", "edgecolor": "#376a8a"},
           medianprops={"color": "#173f5f", "linewidth": 1.5})
ax.set(title="SE uncertainty distributions by TF (sampled values)", ylabel="SE", xlabel="Transcription factor")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
report_figures = ROOT / "reports" / "figures"
report_figures.mkdir(parents=True, exist_ok=True)
fig.savefig(report_figures / "se_uncertainty_by_tf.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)


/tmp/ipykernel_57879/3948067185.py:17: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([se_samples[tf] for tf in TF_CYCLES], labels=list(TF_CYCLES), showfliers=False, patch_artist=True,


The sample sizes vary substantially across TFs. A reproducible streaming sample of SE values is used to compare their uncertainty distributions without loading every row into memory for plotting. The low-count sensitivity analysis retains observations with `ObservedCount ≥ 2`, excluding only single-read sequences. No alternative thresholds were searched. Because count and Affinity are related, filtered and unfiltered results are sensitivity views of the same enrichment data rather than independent replications.

## 7. Shape–Affinity relationships

For each TF and each of 76 canonical tetramer/hexamer predictors, Stage 1 calculated unadjusted and composition-adjusted Pearson and Spearman coefficients. The primary summary below uses all sequences and Pearson correlations. Group summaries first take each TF's median absolute coefficient within a group, then the median of those 11 TF-level summaries; this reports typical association magnitude without allowing groups with more features to dominate.

In [30]:
group_tf = pd.read_csv(SHAPE / "descriptor_group_summary_by_tf.csv")
primary = group_tf[(group_tf.Filter == "All_sequences") & (group_tf.Method == "Pearson") & (group_tf.Adjustment == "Linear_composition_adjusted")]
summary = primary.groupby("Feature_group", as_index=False).agg(
    Median_TF_median_absolute_r=("Median_absolute_correlation", "median"),
    Median_TF_median_signed_r=("Median_correlation", "median"),
    Median_positive_fraction=("Positive_fraction", "median"),
)
display(summary.sort_values("Median_TF_median_absolute_r", ascending=False).style.format({c: "{:.3f}" for c in summary.columns if c != "Feature_group"}))


,Feature_group,Median_TF_median_absolute_r,Median_TF_median_signed_r,Median_positive_fraction
3,Ion-related,0.045,0.004,0.500
2,FCE,0.036,0.005,0.524
0,AVG,0.026,-0.003,0.417
1,Electrostatic,0.014,-0.002,0.333


In [31]:
affinity_corr = pd.read_csv(SHAPE / "affinity_correlations_long.csv")
max_abs = affinity_corr[(affinity_corr.Filter == "All_sequences") & (affinity_corr.Adjustment == "Linear_composition_adjusted")].groupby("Method").Correlation.apply(lambda x: x.abs().max()).rename("Maximum absolute adjusted coefficient")
display(max_abs.to_frame().style.format("{:.3f}"))
display(Image(filename=str(SHAPE / "heatmaps" / "affinity_All_sequences_Pearson_Linear_composition_adjusted.png")))


,Maximum absolute adjusted coefficient
Method,
Pearson,0.249
Spearman,0.223


The linear adjustment uses an intercept and A, C and G frequencies; T is omitted because the four frequencies sum to one. GC is not added because it is C + G. For Pearson, both the descriptor and Affinity are residualised against this design before correlating the residuals. For the complementary rank-residual Spearman estimate, descriptor and Affinity are ranked first and those rank vectors are residualised against the same composition variables on their original frequency scale. This is not conventional partial Spearman correlation.

In [33]:
fit = pd.read_csv(SHAPE / "composition_model_fit.csv")
fit = fit[(fit.Filter == "All_sequences")].pivot(index="TranscriptionFactor", columns="Composition_model", values="Composition_only_Affinity_R_squared")
fit["Quadratic minus linear R²"] = fit["Quadratic"] - fit["Linear"]
display(fit.style.format("{:.3f}"))


Composition_model,Linear,Quadratic,Quadratic minus linear R²
TranscriptionFactor,,,
Gata4,0.224,0.230,0.006
Zfp24,0.116,0.246,0.130
arid3,0.096,0.213,0.117
hoxd1,0.001,0.049,0.048
nkx3,0.342,0.385,0.043
pax3-like,0.150,0.160,0.009
six1,0.259,0.288,0.029
tbx15,0.078,0.094,0.016
telf,0.135,0.140,0.005


The quadratic basis adds the three squared terms and three pairwise interactions in A, C and G. Its in-sample composition-only Affinity R² is a fit diagnostic, not held-out predictive performance. The basis improves R² for all TFs, with improvement varying by dataset; this cautions against treating the linear adjustment as a complete model of composition effects.

In [35]:
linear = affinity_corr[(affinity_corr.Adjustment == "Linear_composition_adjusted") & (affinity_corr.Filter.isin(["All_sequences", "Count_ge_2"]))]
wide = linear.pivot_table(index=["TranscriptionFactor", "Feature", "Method"], columns="Filter", values="Correlation")
sens = []
for method, frame in wide.groupby(level="Method"):
    x, y = frame["All_sequences"], frame["Count_ge_2"]
    sens.append({"Method": method, "Compared TF-feature associations": len(frame), "Same nonzero direction": int(((x * y) > 0).sum()), "Direction agreement": ((x * y) > 0).mean(), "Median absolute coefficient change": (x-y).abs().median()})
display(pd.DataFrame(sens).style.format({"Direction agreement": "{:.1%}", "Median absolute coefficient change": "{:.3f}"}))
display(validation[["TranscriptionFactor", "N_sequences", "Count_ge_2_retained", "Count_ge_2_removed"]])


,Method,Compared TF-feature associations,Same nonzero direction,Direction agreement,Median absolute coefficient change
0,Pearson,836,774,92.6%,0.008
1,Spearman,836,756,90.4%,0.010


,TranscriptionFactor,N_sequences,Count_ge_2_retained,Count_ge_2_removed
0,Gata4,126624,60155,66469
1,Zfp24,406165,226570,179595
2,arid3,273249,144479,128770
3,hoxd1,887634,662991,224643
4,nkx3,963575,848128,115447
5,pax3-like,427916,226515,201401
6,six1,944350,792523,151827
7,tbx15,25991,13599,12392
8,telf,691492,404159,287333
9,uncxb,234129,128267,105862


The count-filter direction diagnostic compares each TF-feature coefficient before and after the filter, separately by correlation method. Zero or sign-changing pairs are counted as disagreement; no tolerance band is applied around zero, so tiny coefficients can count as directionally consistent. The diagnostic is descriptive and does not test statistical stability.

## 8. Conclusions and limitations

Across the 11 TFs, composition-adjusted associations remain observable but are generally modest and vary in direction. Quadratic composition adjustment improves the composition-only fit for every TF and changes some descriptor coefficients, while the low-count filter preserves most signs with small-to-moderate coefficient shifts. These results support continued investigation of descriptor information conditional on the chosen composition models; they do not establish that descriptors improve prediction or are independent of all sequence-composition effects.

A future predictive study would need held-out evaluation, an explicit composition-only baseline, leakage-safe validation, and appropriate handling of TF and sequence dependence. The current EDA makes no causal, binding-affinity Kd, or multiple-comparison significance claims.

## References

1. Rohs R, et al. The role of DNA shape in protein–DNA recognition. *Nature*. 2009;461:1248–1253. doi: [10.1038/nature08473](https://doi.org/10.1038/nature08473).
2. Jolma A, et al. DNA-binding specificities of human transcription factors. *Cell*. 2013;152:327–339. doi: [10.1016/j.cell.2012.12.009](https://doi.org/10.1016/j.cell.2012.12.009).
3. Pedregosa F, et al. Scikit-learn: Machine Learning in Python. *Journal of Machine Learning Research*. 2011;12:2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html.

The project-specific methods and results are documented in `outputs/eda/README.md` and the scripts under `scripts/eda_*.py`.